In [ ]:
import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
q3_data= os.path.join(path, 'Q3_data.csv')
df_data3 = pd.read_csv(q3_data)

print(f"Shape: {df_data3.shape}")


In [ ]:
# Task 2: Write your code here:
df_data3.head()

In [ ]:
# Task 3: Write your code here:
df_data3.info()

In [ ]:
# Task 4: Write your code here:
df_data3.describe()

In [ ]:
# Task 1: Write your code here:
print("Missing values:")
print(df_data3.isnull().sum())

In [ ]:
df_data3.describe()

In [ ]:
categorical_cols = df_data3.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 2: Write your code here:
# Check and remove duplicates if any exist
initial_rows = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
final_rows = df_clean.shape[0]

print(f"Initial number of rows: {initial_rows}")
print(f"Number of rows after removing duplicates: {final_rows}")
print(f"Number of duplicate rows removed: {initial_rows - final_rows}")

In [ ]:
df_clean = df_data3.copy()

# Identify numerical columns (excluding the 'Target' column if it's numerical and shouldn't be imputed)
numerical_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

# Remove 'Target' from numerical_cols if it's there and you don't want to impute its missing values
if 'Target' in numerical_cols:
    numerical_cols.remove('Target')

# Impute missing values in numerical columns with their mean
for col in numerical_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

print(f"After handling missing values: {df_clean.shape}")
print("Missing values after imputation:")
print(df_clean.isnull().sum().sum()) # Print total number of missing values

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Exclude the 'Target' column from scaling
features_to_scale = df_clean.drop(columns=['Target']).columns

scaler = StandardScaler()
df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])

print("Feature scaling applied to numerical features.")
print(df_clean.head())

In [ ]:
# Task 5: Check for target imbalance and state if it is imbalanced or not
# Assuming 'Target' is the column containing the target variable
target_counts = df_clean['Target'].value_counts()
print("Target variable distribution:")
print(target_counts)

# Calculate the percentage of each class
target_percentage = df_clean['Target'].value_counts(normalize=True) * 100
print("\nTarget variable percentage:")
print(target_percentage)

# Determine if it's imbalanced (e.g., if one class is significantly smaller than others, often a ratio like 1:3 or more extreme)
# A common rule of thumb is if the minority class is less than 20-30% of the total.

minority_class_percentage = target_percentage.min()
if minority_class_percentage < 30:
    print(f"\nConclusion: The target variable is imbalanced. The minority class accounts for {minority_class_percentage:.2f}% of the data.")
else:
    print("\nConclusion: The target variable appears to be balanced.")

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df_clean.drop('Target', axis=1)
y = df_clean['Target']

print("Dataset split into features (X) and target (y).")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
!pip install catboost
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np


# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# List to store F1 scores for each fold
f1_scores = []

print("Starting Stratified K-Fold Cross-Validation...")

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold+1}/")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Initialize CatBoostClassifier

    model = CatBoostClassifier(iterations=100,
                               learning_rate=0.1,
                               depth=6,
                               loss_function='Logloss',
                               eval_metric='F1',
                               random_seed=42,
                               verbose=0, # Set to 100 for verbose output every 100 iterations
                               early_stopping_rounds=10 # Early stopping to prevent overfitting
                              )

    # Train the model
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=10, verbose=0)

    # Make predictions on the validation set
    y_pred = model.predict(X_val)

    # Calculate F1 score
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"F1 Score for Fold {fold+1}: {f1:.4f}")

# Calculate and print the averaged F1 score
average_f1_score = np.mean(f1_scores)
print(f"\nAverage F1 Score across all folds: {average_f1_score:.4f}")

In [ ]:
# Task 1: Plot feature importance from your trained model

final_model = CatBoostClassifier(iterations=100,
                               learning_rate=0.1,
                               depth=6,
                               loss_function='Logloss',
                               eval_metric='F1',
                               random_seed=42,
                               verbose=0)

final_model.fit(X, y)

feature_importance = final_model.get_feature_importance()
feature_names = X.columns

# Create a pandas Series for easier manipulation and sorting
importance_df = pd.Series(feature_importance, index=feature_names)

# Sort features by importance
sorted_importance = importance_df.sort_values(ascending=False)

# Plotting feature importances
plt.figure(figsize=(12, 8))
sorted_importance.head(20).plot(kind='barh') # Plot top 20 features
plt.title('Top 20 Feature Importances')
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.gca().invert_yaxis() # Highest importance at the top
plt.show()

print("Feature importance plot displayed.")

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: